In [3]:
pip install openpyxl formulas pandas

   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ------------------------------ --------- 4.7/6.3 MB 23.8 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 20.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [19]:
import os
import glob
from datetime import datetime, timedelta
import win32com.client
import pythoncom
import time
import re

def force_excel_refresh(excel_file_path):
    """
    Force Excel to completely recalculate all formulas and clear caches
    """
    print(f"\nForcing refresh of: {os.path.basename(excel_file_path)}")
    
    pythoncom.CoInitialize()
    
    try:
        # Create Excel instance
        excel = win32com.client.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.AskToUpdateLinks = False
        excel.Calculation = -4105  # xlCalculationAutomatic
        
        # Open workbook
        workbook = excel.Workbooks.Open(excel_file_path)
        
        print(f"  Workbook opened: {workbook.Name}")
        
        # STRATEGY 1: Clear all caches
        try:
            # Clear pivot caches
            for pc in workbook.PivotCaches():
                pc.Refresh()
        except:
            pass
        
        # STRATEGY 2: Force calculation from beginning
        excel.CalculateFullRebuild()
        
        # STRATEGY 3: Calculate each sheet individually
        for sheet in workbook.Sheets:
            try:
                sheet.Calculate()
            except:
                pass
        
        # STRATEGY 4: Force dependencies to recalculate
        # Save, close, and reopen to clear calculation chain
        temp_path = excel_file_path.replace('.xlsx', '_TEMP_REFRESHED.xlsx')
        workbook.SaveAs(temp_path)
        workbook.Close(False)
        
        # Reopen the saved file
        workbook = excel.Workbooks.Open(temp_path)
        
        # STRATEGY 5: Manual recalculation trigger
        # Force cells to recalculate by touching them
        sheet_names = ["DBInsert", "Data", "Sheet1", "Sheet2", "Sheet3"]
        for sheet_name in sheet_names:
            try:
                sheet = workbook.Sheets(sheet_name)
                # Force recalc by reading a cell that likely has formulas
                if sheet.UsedRange.Count > 0:
                    # Read first cell to trigger calculation
                    _ = sheet.Cells(1, 1).Value
                    sheet.Calculate()
            except:
                pass
        
        # Final calculation
        excel.CalculateUntilAsyncQueriesDone()
        
        # Wait longer for complex calculations
        time.sleep(5)
        
        # Save final version
        final_path = excel_file_path.replace('.xlsx', '_FINAL_REFRESHED.xlsx')
        workbook.SaveAs(final_path)
        workbook.Close(True)
        
        excel.Quit()
        
        # Replace original with refreshed file
        os.remove(excel_file_path)
        os.rename(final_path, excel_file_path)
        os.remove(temp_path) if os.path.exists(temp_path) else None
        
        print(f"  ✓ Complete refresh cycle completed")
        return True
        
    except Exception as e:
        print(f"  ✗ Error during refresh: {str(e)}")
        try:
            excel.Quit()
        except:
            pass
        return False
    finally:
        pythoncom.CoUninitialize()

def deep_read_excel_values(excel_file_path):
    """
    Read Excel values using multiple methods to ensure fresh data
    """
    print(f"  Reading values from: {os.path.basename(excel_file_path)}")
    
    sql_statements = []
    
    # METHOD 1: Try with openpyxl first
    try:
        from openpyxl import load_workbook
        
        workbook = load_workbook(excel_file_path, data_only=True)
        
        if "DBInsert" not in workbook.sheetnames:
            print(f"    'DBInsert' sheet not found")
            workbook.close()
            return []
        
        sheet = workbook["DBInsert"]
        
        # Read specific rows
        for row_num in range(72, 489):
            try:
                row = sheet[row_num]
                if row and row[0].value:
                    value = str(row[0].value).strip()
                    if value:
                        sql_statements.append(value)
            except:
                continue
        
        workbook.close()
        
        if sql_statements:
            print(f"    Read {len(sql_statements)} statements via openpyxl")
            return sql_statements
            
    except Exception as e:
        print(f"    openpyxl read failed: {str(e)}")
    
    # METHOD 2: Try pandas as fallback
    try:
        import pandas as pd
        
        # Read entire column A
        df = pd.read_excel(excel_file_path, 
                          sheet_name='DBInsert',
                          usecols=[0],
                          header=None)
        
        # Filter rows 72-488 (0-indexed: 71-487)
        df_filtered = df.iloc[71:488] if len(df) > 71 else pd.DataFrame()
        
        for value in df_filtered.iloc[:, 0].dropna():
            sql_statements.append(str(value).strip())
        
        print(f"    Read {len(sql_statements)} statements via pandas")
        return sql_statements
        
    except Exception as e:
        print(f"    pandas read failed: {str(e)}")
    
    # METHOD 3: Direct COM reading (most likely to get fresh values)
    pythoncom.CoInitialize()
    try:
        excel = win32com.client.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        
        workbook = excel.Workbooks.Open(excel_file_path)
        
        # Force immediate calculation
        excel.Calculation = -4105  # Automatic
        workbook.Application.CalculateFullRebuild()
        
        sheet = workbook.Sheets("DBInsert")
        
        # Read cells directly via COM
        for row_num in range(72, 489):
            try:
                cell_value = sheet.Cells(row_num, 1).Value
                if cell_value:
                    sql_statements.append(str(cell_value).strip())
            except:
                continue
        
        workbook.Close(False)
        excel.Quit()
        
        print(f"    Read {len(sql_statements)} statements via COM")
        return sql_statements
        
    except Exception as e:
        print(f"    COM read failed: {str(e)}")
        try:
            excel.Quit()
        except:
            pass
        return []
    finally:
        pythoncom.CoUninitialize()

def verify_refresh_needed(excel_file_path):
    """
    Check if file needs refresh by looking for old values
    """
    try:
        from openpyxl import load_workbook
        
        workbook = load_workbook(excel_file_path, data_only=True)
        
        if "DBInsert" not in workbook.sheetnames:
            workbook.close()
            return True  # Refresh anyway
        
        sheet = workbook["DBInsert"]
        
        # Sample a few cells to check for old values
        sample_cells = [(72, 1), (100, 1), (200, 1)]
        
        for row, col in sample_cells:
            try:
                cell_value = str(sheet.cell(row=row, column=col).value or "")
                # Check for old ticker name
                if "UTI ASSET MANAGEMENT COMPANY LTD" in cell_value:
                    print(f"    Found old value in cell ({row},{col}), refresh needed")
                    workbook.close()
                    return True
            except:
                continue
        
        workbook.close()
        return False
        
    except:
        return True  # If we can't check, assume refresh needed

def generate_db_insert_script_enhanced(base_folder, date_range="All"):
    """
    Main function with enhanced refresh logic
    """
    print("=" * 80)
    print("ENHANCED DB INSERT SCRIPT GENERATOR")
    print("=" * 80)
    
    # Find latest folder
    subfolders = [f.path for f in os.scandir(base_folder) if f.is_dir()]
    if not subfolders:
        print("No subfolders found")
        return None
    
    latest_folder = max(subfolders, key=os.path.getmtime)
    print(f"Processing folder: {os.path.basename(latest_folder)}")
    
    # Create output file
    timestamp = datetime.now().strftime("%Y%m%d%H%M")
    output_file = os.path.join(latest_folder, f"{timestamp}_DBInsertScript.sql")
    
    # Find Excel files
    excel_files = []
    for file in os.listdir(latest_folder):
        if file.endswith('.xlsx') and not file.startswith(('$', '~')):
            full_path = os.path.join(latest_folder, file)
            
            # Date filtering
            mod_time = datetime.fromtimestamp(os.path.getmtime(full_path))
            if date_range == "Today" and mod_time.date() != datetime.now().date():
                continue
            elif date_range == "TodayAndYesterday":
                yesterday = datetime.now().date() - timedelta(days=1)
                if mod_time.date() not in [datetime.now().date(), yesterday]:
                    continue
            
            excel_files.append(full_path)
    
    excel_files.sort(key=os.path.getmtime, reverse=True)
    print(f"Found {len(excel_files)} files to process")
    print("-" * 80)
    
    # Process files
    total_statements = 0
    with open(output_file, 'w', encoding='utf-8') as writer:
        for idx, excel_file in enumerate(excel_files, 1):
            filename = os.path.basename(excel_file)
            print(f"\n[{idx}/{len(excel_files)}] Processing: {filename}")
            
            # Check if refresh is needed
            if verify_refresh_needed(excel_file):
                print(f"  Refresh required for {filename}")
                if not force_excel_refresh(excel_file):
                    print(f"  ⚠ Could not refresh {filename}, using existing values")
            else:
                print(f"  ✓ File already appears up-to-date")
            
            # Read values with multiple fallback methods
            sql_statements = deep_read_excel_values(excel_file)
            
            if sql_statements:
                # Validate statements contain new ticker names
                old_ticker_found = False
                new_ticker_found = False
                
                for stmt in sql_statements:
                    if "UTI ASSET MANAGEMENT COMPANY LTD" in stmt:
                        old_ticker_found = True
                    if "UTIAMC" in stmt:
                        new_ticker_found = True
                
                if old_ticker_found and new_ticker_found:
                    print(f"  ⚠ Mixed old and new ticker names found!")
                elif old_ticker_found:
                    print(f"  ⚠ Old ticker names still present after refresh!")
                
                # Write to file
                for stmt in sql_statements:
                    writer.write(stmt + "\n")
                writer.write("\n")
                writer.flush()
                
                total_statements += len(sql_statements)
                print(f"  ✓ Added {len(sql_statements)} statements")
            else:
                print(f"  ✗ No statements found")
    
    print("\n" + "=" * 80)
    print(f"COMPLETE: Generated {total_statements} SQL statements")
    print(f"Output: {output_file}")
    
    return output_file

# Quick manual fix function for specific files
def manual_fix_ticker_in_excel(excel_file_path, old_ticker, new_ticker):
    """
    Manually replace ticker names in Excel file
    """
    print(f"\nManually fixing {os.path.basename(excel_file_path)}")
    print(f"  Replacing: '{old_ticker}' -> '{new_ticker}'")
    
    pythoncom.CoInitialize()
    try:
        excel = win32com.client.Dispatch("Excel.Application")
        excel.Visible = True  # Show for manual inspection
        excel.DisplayAlerts = False
        
        workbook = excel.Workbooks.Open(excel_file_path)
        
        # Search and replace in all sheets
        for sheet in workbook.Sheets:
            try:
                used_range = sheet.UsedRange
                # Replace in formulas
                used_range.Replace(What=old_ticker, 
                                  Replacement=new_ticker,
                                  LookAt=1,  # xlWhole
                                  SearchOrder=1,  # xlByRows
                                  MatchCase=False)
                
                # Replace in values
                used_range.Replace(What=old_ticker,
                                  Replacement=new_ticker,
                                  LookAt=1,
                                  SearchOrder=1,
                                  MatchCase=False,
                                  SearchFormat=False,
                                  ReplaceFormat=False)
                
                print(f"  Processed sheet: {sheet.Name}")
                
            except Exception as e:
                print(f"  Error in sheet {sheet.Name}: {str(e)}")
        
        # Save and close
        workbook.Save()
        workbook.Close()
        excel.Quit()
        
        print(f"  ✓ Manual fix completed")
        
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        try:
            excel.Quit()
        except:
            pass
    finally:
        pythoncom.CoUninitialize()

def batch_fix_tickers(folder_path):
    """
    Batch fix ticker names in all Excel files
    """
    excel_files = [f for f in os.listdir(folder_path) 
                  if f.endswith('.xlsx') and not f.startswith(('$', '~'))]
    
    print(f"Found {len(excel_files)} files to fix")
    
    for excel_file in excel_files:
        full_path = os.path.join(folder_path, excel_file)
        manual_fix_ticker_in_excel(
            full_path,
            "UTI ASSET MANAGEMENT COMPANY LTD",
            "UTIAMC"
        )

if __name__ == "__main__":
    base_path = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels"
    
    # Option 1: Generate script with enhanced refresh
    # generate_db_insert_script_enhanced(base_path, "All")
    
    # Option 2: First manually fix tickers, then generate
    # Uncomment below to run manual fix first:
    
    # print("STEP 1: Manually fixing ticker names...")
    # subfolders = [f.path for f in os.scandir(base_path) if f.is_dir()]
    # if subfolders:
    #     latest_folder = max(subfolders, key=os.path.getmtime)
    #     batch_fix_tickers(latest_folder)
    
    print("\nSTEP 2: Generating DB insert script...")
    generate_db_insert_script_enhanced(base_path, "All")


STEP 2: Generating DB insert script...
ENHANCED DB INSERT SCRIPT GENERATOR
Processing folder: 2026Q2
Found 771 files to process
--------------------------------------------------------------------------------

[1/771] Processing: WEWORK_FY26Q2.xlsx


C:\Users\Sudhir Kulaye\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Sparkline Group extension is not supported and will be removed
  ws_parser.bind_all()


  ✓ File already appears up-to-date
  Reading values from: WEWORK_FY26Q2.xlsx
    Read 40 statements via openpyxl
  ✓ Added 40 statements

[2/771] Processing: VIKRAMSOLR_FY26Q2.xlsx
  ✓ File already appears up-to-date
  Reading values from: VIKRAMSOLR_FY26Q2.xlsx
    Read 46 statements via openpyxl
  ✓ Added 46 statements

[3/771] Processing: VENUSPIPES_FY26Q2.xlsx
  ✓ File already appears up-to-date
  Reading values from: VENUSPIPES_FY26Q2.xlsx
    Read 74 statements via openpyxl
  ✓ Added 74 statements

[4/771] Processing: URBANCO_FY26Q2.xlsx
  ✓ File already appears up-to-date
  Reading values from: URBANCO_FY26Q2.xlsx
    Read 42 statements via openpyxl
  ✓ Added 42 statements

[5/771] Processing: UFBL_FY26Q2.xlsx
  ✓ File already appears up-to-date
  Reading values from: UFBL_FY26Q2.xlsx
    Read 80 statements via openpyxl
  ✓ Added 80 statements

[6/771] Processing: TSFINV_FY26Q2.xlsx
  ✓ File already appears up-to-date
  Reading values from: TSFINV_FY26Q2.xlsx
    Read 68 statem

C:\Users\Sudhir Kulaye\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Sparkline Group extension is not supported and will be removed
  ws_parser.bind_all()


  ✓ File already appears up-to-date
  Reading values from: TIMETECHNO_FY26Q2.xlsx
    pandas read failed: single positional indexer is out-of-bounds
    COM read failed: (-2147352567, 'Exception occurred.', (0, 'Microsoft Excel', 'Open method of Workbooks class failed', 'xlmain11.chm', 0, -2146827284), None)
  ✗ No statements found

[406/771] Processing: SKFINDIA_FY26Q2.xlsx


...

  ✓ File already appears up-to-date
  Reading values from: ALOKINDS_FY26Q2.xlsx
    pandas read failed: single positional indexer is out-of-bounds
    Read 124 statements via COM
  ✓ Added 124 statements

[768/771] Processing: IRFC_FY26Q2.xlsx


C:\Users\Sudhir Kulaye\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Sparkline Group extension is not supported and will be removed
  ws_parser.bind_all()


  ✓ File already appears up-to-date
  Reading values from: IRFC_FY26Q2.xlsx
    pandas read failed: single positional indexer is out-of-bounds
    Read 124 statements via COM
  ✓ Added 124 statements

[769/771] Processing: MAHABANK_FY26Q2.xlsx


C:\Users\Sudhir Kulaye\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Sparkline Group extension is not supported and will be removed
  ws_parser.bind_all()


  ✓ File already appears up-to-date
  Reading values from: MAHABANK_FY26Q2.xlsx
    Read 132 statements via openpyxl
  ✓ Added 132 statements

[770/771] Processing: IREDA_FY26Q2.xlsx
  ✓ File already appears up-to-date
  Reading values from: IREDA_FY26Q2.xlsx
    Read 78 statements via openpyxl
  ✓ Added 78 statements

[771/771] Processing: ELECON_FY26Q2.xlsx
  ✓ File already appears up-to-date
  Reading values from: ELECON_FY26Q2.xlsx
    Read 110 statements via openpyxl
  ✓ Added 110 statements

COMPLETE: Generated 73786 SQL statements
Output: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\2026Q2\202512132119_DBInsertScript.sql
